In [ ]:
import warnings
import pandas as pd
from pathlib import Path
from datasets import Dataset
from utils.bq import get_client, query_to_df
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

SQL_DIR = Path().resolve().parent / "sql_scripts"
client, PROJECT_NAME = get_client()

In [ ]:
# Radiology Reports — ED Visit + Hospital Ward Window (action records + state context) ---
# Imaging studies ordered during the ED stay and subsequent hospital ward stay (pre-ICU).
# hadm_id carried through from the radiology table — NULL for ED-only patients, populated for admitted.
# Filter note_type = 'RR' for primary reports only (exclude 'AR' addenda).
# exam_name and cpt_code pulled from radiology_detail to identify imaging modality.
#
# Window logic:
#   - ED-only patients (hadm_id IS NULL):  ed_intime → ed_outtime
#   - Admitted patients (hadm_id IS NOT NULL): ed_intime → first_icu_intime (or dischtime if no ICU)

query_radiology = (SQL_DIR / "radiology_data.sql").read_text().format(PROJECT_NAME=PROJECT_NAME)

print("Running Query 7: Radiology reports...")
df_radiology = query_to_df(client, query_radiology)
print(f"Shape: {df_radiology.shape}")

print(f"Unique ED visits with imaging: {df_radiology['ed_stay_id'].nunique():,}")
print(f"Records with hadm_id (admitted patients): {df_radiology['hadm_id'].notna().sum():,}")
print(f"Records without hadm_id (ED-only patients): {df_radiology['hadm_id'].isna().sum():,}")
print(f"\nTop exam types:\n{df_radiology['exam_name'].value_counts().head(10)}")
df_radiology.head()

In [ ]:
DESCRIPTION = (
    "Radiology report text from mimiciv_note.radiology for cohort patients. "
    "Covers all imaging modalities (CXR, CT, MRI, ultrasound, etc.). Primary reports only (note_type=RR). "
    "Window covers ED arrival through hospital ward stay (capped at ICU transfer if applicable). "
    "hadm_id is NULL for ED-only patients and populated for admitted patients. "
    "exam_name and cpt_code included from radiology_detail to identify imaging modality."
)

ds = Dataset.from_pandas(df_radiology, split="radiology_details")
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="radiology_details", data_dir='radiology')
push_dataset_card("ADS599-Capstone/raw_data", config_name="radiology_details", split="radiology_details", data_dir="radiology", description=DESCRIPTION)
print("Pushed radiology data to HuggingFace Hub.")